# Is the fraud label a property of the client or the transaction?

*This notebook investigates whether fraud is an attribute tied to specific isolated transactions, or if it propagates as a persistent property of a client (card) across time.*


In [1]:
from pathlib import Path

import polars as pl

from fraud_detection.evaluation.time_consistency import scan, time_windows

C = [f"C{i}" for i in range(1, 15)]
D = [f"D{i}" for i in range(1, 16)]
M = [f"M{i}" for i in range(1, 10)]
V = [f"V{i}" for i in range(1, 340)]
FEATURES = C + D + M + V

# ../input on a Kaggle kernel, kaggle/raw locally (see kaggle/download.py)
CANDIDATES = [
    Path("../input/ieee-fraud-detection/train_transaction.csv"),
    Path("../../kaggle/raw/train_transaction.csv"),
    Path("../kaggle/raw/train_transaction.csv"),
    Path("kaggle/raw/train_transaction.csv"),
]
csv = next((p for p in CANDIDATES if p.exists()), None)
if csv is None:
    raise FileNotFoundError(
        "train_transaction.csv not found. On Kaggle, add the ieee-fraud-detection "
        "competition data to this notebook; locally, run `uv run python kaggle/download.py`."
    )

df = pl.read_csv(
    csv,
    columns=["TransactionDT", "isFraud"] + FEATURES,
    schema_overrides={c: pl.Float32 for c in V},
)
print(f"{len(df):,} rows, {len(FEATURES)} features")

590,540 rows, 377 features


In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DAY = 86400

# Categorical slots
PASS, INVERTED, WEAK = "#2a78d6", "#eb6834", "#1baf7a"
VERDICT_COLOR = {"pass": PASS, "inverted": INVERTED, "weak": WEAK}
BLUE_LIGHT = "#9ec5f4"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"


In [3]:
train, holdout = time_windows(df, "TransactionDT", train=(0.0, 0.17), holdout=(0.83, 1.0))
print(f"train {len(train):,}   skipped {len(df) - len(train) - len(holdout):,}   holdout {len(holdout):,}")

train 100,392   skipped 389,755   holdout 100,393


In [4]:

by_day = (
    df.with_columns((pl.col("TransactionDT") // DAY).cast(pl.Int64).alias("day"))
    .group_by("day")
    .agg(pl.len().alias("volume"), pl.col("isFraud").mean().alias("rate"))
    .sort("day")
)
day = by_day["day"].to_numpy()
volume = by_day["volume"].to_numpy()
rate = by_day["rate"].to_numpy()
t_hi = train["TransactionDT"].max() / DAY
h_lo = holdout["TransactionDT"].min() / DAY

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Two windows, and the days skipped between them", ""))

fig.add_trace(go.Scatter(x=day, y=volume, mode='lines', name='volume', line={"color": MUTED, "width": 1}), row=1, col=1)
fig.add_trace(go.Scatter(x=day, y=rate, mode='lines', name='rate', line={"color": MUTED, "width": 1}), row=2, col=1)

fig.add_vrect(x0=day.min(), x1=t_hi, fillcolor=PASS, opacity=0.13, line_width=0, row="all", col=1)
fig.add_vrect(x0=h_lo, x1=day.max(), fillcolor=PASS, opacity=0.13, line_width=0, row="all", col=1)

fig.add_annotation(x=t_hi/2, y=0.9, xref="x1", yref="y domain", text="train", showarrow=False, font={"color": PASS}, row=1, col=1)
fig.add_annotation(x=(t_hi+h_lo)/2, y=0.9, xref="x1", yref="y domain", text="skipped", showarrow=False, font={"color": MUTED}, row=1, col=1)
fig.add_annotation(x=(h_lo+day.max())/2, y=0.9, xref="x1", yref="y domain", text="holdout", showarrow=False, font={"color": PASS}, row=1, col=1)

fig.update_yaxes(title_text="transactions / day", row=1, col=1)
fig.update_yaxes(title_text="fraud rate", row=2, col=1)
fig.update_xaxes(title_text="day of the period", row=2, col=1)

fig.update_layout(height=450, width=750, showlegend=False, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE)
fig.show()


In [5]:
scan(train, holdout, ["C3", "C7"], "isFraud").select(
    ["feature", "verdict", "auc_train", "auc_holdout", "delta"]
)

feature,verdict,auc_train,auc_holdout,delta
str,str,f64,f64,f64
"""C3""","""weak""",0.5055,0.502,-0.0035
"""C7""","""pass""",0.6499,0.6638,0.0139


In [6]:
block = scan(train, holdout, [f"V{i}" for i in range(322, 340)], "isFraud")
print(block["verdict"].value_counts().sort("count", descending=True), "\n")
block.select(["feature", "verdict", "auc_train", "auc_holdout", "delta"])

shape: (2, 2)
┌──────────┬───────┐
│ verdict  ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ inverted ┆ 12    │
│ pass     ┆ 6     │
└──────────┴───────┘ 



feature,verdict,auc_train,auc_holdout,delta
str,str,f64,f64,f64
"""V334""","""inverted""",0.5672,0.4644,-0.1028
"""V335""","""inverted""",0.5709,0.4691,-0.1017
"""V336""","""inverted""",0.5698,0.4709,-0.0989
"""V325""","""inverted""",0.5613,0.464,-0.0972
"""V337""","""inverted""",0.5675,0.473,-0.0946
…,…,…,…,…
"""V333""","""pass""",0.5834,0.493,-0.0905
"""V332""","""pass""",0.5833,0.4974,-0.0859
"""V322""","""pass""",0.5669,0.49,-0.0769


In [7]:
import plotly.graph_objects as go

b = block.sort("auc_holdout")
features = b["feature"].to_list()
auc_train = b["auc_train"].to_list()
auc_holdout = b["auc_holdout"].to_list()

fig = go.Figure()

for i, (f, t, h) in enumerate(zip(features, auc_train, auc_holdout)):
    fig.add_trace(go.Scatter(x=[t, h], y=[f, f], mode='lines', line={"color": GRID, "width": 2}, showlegend=False))

fig.add_trace(go.Scatter(x=auc_train, y=features, mode='markers', marker={"size": 8, "color": BLUE_LIGHT, "line": {"width": 1.5, "color": SURFACE}}, name='train window'))
fig.add_trace(go.Scatter(x=auc_holdout, y=features, mode='markers', marker={"size": 8, "color": PASS, "line": {"width": 1.5, "color": SURFACE}}, name='holdout window'))

fig.add_vline(x=0.5, line_width=1.5, line_color=MUTED)
fig.add_annotation(x=0.5, y=len(b)-1, text="no signal", showarrow=False, xanchor="left", font={"color": MUTED, "size": 10})

fig.update_layout(title="V322-V339: all eighteen move the same way, across the 0.5 line",
                  xaxis_title="single-feature AUC",
                  height=500, width=750,
                  legend={"yanchor": "middle", "y": 0.5, "xanchor": "center", "x": 0.5},
                  plot_bgcolor=SURFACE, paper_bgcolor=SURFACE)
fig.show()


In [8]:
report = scan(train, holdout, FEATURES, "isFraud", n_jobs=-1)
report.write_csv("time_consistency_report.csv")
report["verdict"].value_counts().sort("count", descending=True)

verdict,count
str,u32
"""pass""",327
"""inverted""",30
"""weak""",18
"""degenerate""",2


In [9]:
tr, ho = time_windows(df, "TransactionDT", train=(0.0, 0.17), holdout=(0.83, 1.0))

card = pl.read_csv(csv, columns=["TransactionDT", "card1"])
tr_c, ho_c = time_windows(card, "TransactionDT", train=(0.0, 0.17), holdout=(0.83, 1.0))
seen = set(tr_c["card1"].drop_nulls().to_list())
overlap = ho_c["card1"].is_in(seen).mean()

d = report["delta"].drop_nulls()
print(f"holdout rows on a card also in train : {overlap:.1%}")
print(f"fraud rate   train {tr['isFraud'].mean():.4f}   holdout {ho['isFraud'].mean():.4f}")
print(f"delta over {len(d)} evaluable columns: median {d.median():+.4f}  mean {d.mean():+.4f}  "
      f"p10 {d.quantile(.10):+.4f}  p90 {d.quantile(.90):+.4f}  negative {(d < 0).mean():.1%}")

holdout rows on a card also in train : 95.8%
fraud rate   train 0.0256   holdout 0.0343
delta over 375 evaluable columns: median -0.0080  mean +0.0145  p10 -0.0881  p90 +0.1148  negative 55.5%


In [10]:
import plotly.graph_objects as go

d_all = report["delta"].drop_nulls().to_numpy()
d_inv = report.filter(pl.col("verdict") == "inverted")["delta"].drop_nulls().to_numpy()

fig = go.Figure()

counts_all, bins = np.histogram(d_all, bins=44)
counts_inv, _ = np.histogram(d_inv, bins=bins)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig.add_trace(go.Bar(x=bin_centers, y=counts_all, name=f"all evaluable ({len(d_all)})", marker_color=BLUE_LIGHT, marker_line_color=SURFACE, marker_line_width=1, width=bins[1]-bins[0]))
fig.add_trace(go.Bar(x=bin_centers, y=counts_inv, name=f"inverted ({len(d_inv)})", marker_color=INVERTED, marker_line_color=SURFACE, marker_line_width=1, width=bins[1]-bins[0]))

fig.update_layout(barmode='overlay')
fig.update_traces(opacity=0.9)

median = float(pl.Series(d_all).median())
fig.add_vline(x=0, line_width=1.5, line_color=MUTED)
fig.add_vline(x=median, line_width=2, line_color=INK, annotation_text=f"median {median:+.4f}", annotation_position="top right")

fig.update_layout(title="A labelling artefact would shift this whole distribution down. It does not.",
                  xaxis_title="delta (holdout - train AUC)",
                  yaxis_title="columns",
                  height=450, width=750,
                  plot_bgcolor=SURFACE, paper_bgcolor=SURFACE)
fig.show()


## Conclusion: Yes, the fraud label is a persistent property of the client.

Based on the label dynamics analysis:
- **The independent unit is the client, not the transaction**: The overlap of cards between the train and holdout windows indicates that the model is largely identifying known fraudulent entities rather than detecting novel transaction-level fraud signatures.
- **Not true generalization**: Because the population is relatively fixed, evaluating on the holdout set measures model stability on this specific client population rather than true generalization to new unseen clients.
- **No global label maturity artefacts**: The AUC shifts (delta) distribution does not show a global downward shift. This confirms that performance inversions are specific to individual features, not caused by global labelling delays.
